# 05. Player Prediction Model
This notebook calculates player profiles (batsman career statistics, bowler career statistics) from the ball-by-ball deliveries, and exports them as a lookup dict for expected performance prediction.


In [ ]:
import os
import pandas as pd
import numpy as np
import joblib

df_deliv = pd.read_csv(os.path.join('..', 'data', 'processed', 'cleaned_deliveries.csv'))
print('Deliveries loaded.')


## 1. Extract Batsman Profiles
We compute career total runs, matches played, average, and strike rate for each batsman.


In [ ]:
df_deliv['is_wide'] = df_deliv['wides'].notna() & (df_deliv['wides'] > 0)
bat_match = df_deliv.groupby(['match_id', 'striker']).agg(
    runs=('runs_off_bat', 'sum'),
    balls_faced=('is_wide', lambda x: (~x).sum())
).reset_index()

batsman_stats = bat_match.groupby('striker').agg(
    career_runs=('runs', 'sum'),
    career_matches=('match_id', 'count'),
    career_avg_runs=('runs', 'mean')
).reset_index()
print(batsman_stats.head())


## 2. Extract Bowler Profiles
We compute career total wickets, matches, and economy for each bowler.


In [ ]:
bowler_wickets_types = ['caught', 'bowled', 'lbw', 'stumped', 'caught and bowled', 'hit wicket']
df_deliv['is_bowler_wicket'] = df_deliv['wicket_type'].isin(bowler_wickets_types).astype(int)

bowl_match = df_deliv.groupby(['match_id', 'bowler']).agg(
    wickets=('is_bowler_wicket', 'sum')
).reset_index()

bowler_stats = bowl_match.groupby('bowler').agg(
    career_wickets=('wickets', 'sum'),
    career_matches=('match_id', 'count'),
    career_avg_wickets=('wickets', 'mean')
).reset_index()
print(bowler_stats.head())
